# KORMo 1B — 벤치마크 평가 (Colab)

`01_kormo_1B_pretrain_colab_a100.ipynb`로 학습해 Drive에 저장한 모델(`final/`)을
[lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness)로 평가합니다 —
HF Open LLM Leaderboard·Open Ko-LLM Leaderboard가 쓰던 것과 같은 도구입니다.

**전제**: Drive `MyDrive/kormo-1B-PT/final/`에 학습된 모델 존재 (1번 노트북 완주)

**런타임**: L4 GPU면 충분합니다 (A100 불필요 — 평가는 forward만 하므로)

구성:
1. **Held-out perplexity** — 학습에 쓰지 않은 한국어 위키 텍스트에 대한 PPL. 튜토리얼 스케일에서 가장 민감한 지표
2. **KoBEST + HAE-RAE** — loglikelihood 기반이라 작은 모델도 측정 가능 (~10분)
3. **KMMLU** (선택) — 45과목 전문지식. 오래 걸리고 이 스케일에선 랜덤 수준이 예상되므로 기본 off

In [ ]:
!nvidia-smi -L

## 0. 환경 준비 + 모델 로드

평가는 문서 단위 forward라 패킹 마스크가 필요 없으므로 `flex_attention` 대신 **sdpa**로 로드합니다 (더 빠르고 컴파일 오버헤드 없음).

In [ ]:
import os
# HF Xet 스토리지 우회 — Colab 익명 접근 시 cas-server 401 오류 방지 (일반 HTTP로 폴백)
os.environ['HF_HUB_DISABLE_XET'] = '1'

!git clone -q https://github.com/MLP-Lab/KORMo-tutorial.git /content/KORMo-tutorial
%pip install -q "transformers==4.57.1" "datasets>=4.1.1" lm-eval

import sys, json
sys.path.append('/content/KORMo-tutorial/src')

# (선택) HF 토큰 — Colab 보안 비밀에 HF_TOKEN 등록 시 자동 사용. gated 데이터셋 접근에도 필요
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print("HF_TOKEN 로드됨")
except Exception:
    pass  # 토큰 없이도 공개 데이터셋은 동작

In [ ]:
import torch
from google.colab import drive
from transformers import AutoTokenizer
from kormo.model._modeling_kormo import KORMoForCausalLM

drive.mount('/content/drive')

FINAL_DIR = '/content/drive/MyDrive/kormo-1B-PT/final'
assert os.path.isdir(FINAL_DIR), (
    f"{FINAL_DIR} 가 없습니다 — 1번 노트북(사전학습)을 먼저 완주해서 final/을 만들어야 합니다."
)

model = KORMoForCausalLM.from_pretrained(
    FINAL_DIR, dtype=torch.bfloat16, _attn_implementation='sdpa',
).to('cuda').eval()

try:
    tokenizer = AutoTokenizer.from_pretrained(FINAL_DIR)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained('KORMo-Team/KORMo-tokenizer')
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"로드 완료: {sum(p.numel() for p in model.parameters())/1e9:.2f}B params")

## 1. Held-out perplexity (한국어 위키)

학습 데이터(KORMo-tutorial-datasets)에 없는 한국어 위키백과 문서 200개에 대해 PPL을 계산합니다.

해석 기준:
- 무작위 초기화 모델: PPL ≈ vocab 크기(125,184) 수준
- 학습이 조금이라도 됐다면: 수백~수천 대로 하락
- 추가 학습(continue)을 반복하며 **이 숫자가 계속 내려가는지**가 튜토리얼 스케일에서 가장 좋은 진행 지표입니다

In [ ]:
import datasets, itertools, math

wiki = datasets.load_dataset('wikimedia/wikipedia', '20231101.ko', split='train', streaming=True)
texts = [ex['text'] for ex in itertools.islice(wiki, 500) if len(ex['text']) > 500][:200]
print(f"평가 문서: {len(texts)}개")

total_nll, total_tokens = 0.0, 0
with torch.no_grad():
    for text in texts:
        ids = tokenizer.encode(text, return_tensors='pt')[:, :2048].to('cuda')
        if ids.shape[1] < 16:
            continue
        loss = model(input_ids=ids, labels=ids).loss  # 토큰 평균 NLL
        n = ids.shape[1] - 1
        total_nll += loss.item() * n
        total_tokens += n

ppl = math.exp(total_nll / total_tokens)
print(f"\nHeld-out PPL (한국어 위키, {total_tokens:,} 토큰): {ppl:,.1f}")

## 2. KoBEST + HAE-RAE (lm-evaluation-harness)

객관식 벤치마크는 생성이 아니라 **선택지별 loglikelihood 비교**로 채점되므로, 옹알이 단계의 base 모델도 평가할 수 있습니다.

무작위 기준선 (이 값 근처면 "아직 못 푸는 것", 유의미하게 위면 학습 신호):

| 태스크 | 방식 | 랜덤 기준선 |
|---|---|---|
| kobest_boolq / copa / wic / sentineg | 2지선다 | 0.50 |
| kobest_hellaswag | 4지선다 | 0.25 |
| haerae (5개 하위 태스크) | 5지선다 | ~0.20 |

In [ ]:
import lm_eval
from lm_eval.models.huggingface import HFLM

lm = HFLM(pretrained=model, tokenizer=tokenizer, batch_size=16)

results = lm_eval.simple_evaluate(
    model=lm,
    tasks=['kobest', 'haerae'],   # 그룹명 — kobest 5종 + haerae 5종 전체
    num_fewshot=0,
)
print(lm_eval.utils.make_table(results))

## 3. (선택) KMMLU — 45과목 전문지식

`RUN_KMMLU = True`로 바꾸면 실행됩니다. 35k 문항 × 4지선다라 L4에서 1시간 이상 걸리고,
튜토리얼 스케일 모델은 0.25(랜덤) 부근이 예상되므로 기본은 꺼져 있습니다.
KORMo-10B 논문 점수와 비교하려면 논문과 같은 5-shot 설정을 쓰세요.

In [ ]:
RUN_KMMLU = False

if RUN_KMMLU:
    kmmlu_results = lm_eval.simple_evaluate(
        model=lm,
        tasks=['kmmlu'],
        num_fewshot=5,   # KORMo 논문 설정
    )
    print(lm_eval.utils.make_table(kmmlu_results))

## 4. 결과 저장

결과를 Drive에 JSON으로 남깁니다. continue 모드로 추가 학습을 반복할 때
이 파일들을 비교하면 학습량 대비 성능 곡선을 그릴 수 있습니다.

In [ ]:
from datetime import datetime

EVAL_DIR = '/content/drive/MyDrive/kormo-1B-PT/evals'
os.makedirs(EVAL_DIR, exist_ok=True)

stamp = datetime.now().strftime('%Y%m%d_%H%M')
summary = {
    'timestamp': stamp,
    'heldout_ppl_ko_wiki': ppl,
    'benchmarks': {k: v for k, v in results['results'].items()},
}
if RUN_KMMLU:
    summary['kmmlu'] = {k: v for k, v in kmmlu_results['results'].items()}

out_path = f'{EVAL_DIR}/eval_{stamp}.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)
print("저장:", out_path)